> **STATUS: NOT EXECUTED.** This notebook is prepared for Google Colab (open internet / GPU) and has **not** been run. No results from it appear in the paper or README. See `01_full_analysis.ipynb` for the executed analysis.

# Misinformation Classification — DistilBERT Fine-Tuning

This notebook fine-tunes DistilBERT on the PolitiFact fake/real title data,
to compare against the TF-IDF + Logistic Regression/SVM baseline already
built locally (87.7% accuracy, F1 = 0.854).

**Why Colab and not the local sandbox:** the local project sandbox has a
restricted network allowlist that blocks Hugging Face's model hub, so
pretrained weights can't be downloaded there. Colab has open internet and a
free GPU, which also makes training much faster than CPU-only fine-tuning.

**What you need:** upload `processed_politifact.csv` (from this project's
`data/` folder) to this Colab's file browser before running, or mount your
Google Drive if you've saved it there.

Run cells top to bottom. GPU runtime recommended (Runtime > Change runtime type > GPU).


In [ ]:
# 1. Install/confirm packages (Colab usually has these, but pin versions for reproducibility)
!pip install transformers datasets scikit-learn -q

import torch
print("Torch version:", torch.__version__, "| CUDA:", torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)


In [ ]:
# 2. Load the processed PolitiFact data
# Upload processed_politifact.csv via the Colab file browser (left sidebar) first,
# or adjust the path if you mounted Google Drive.
import pandas as pd

df = pd.read_csv('processed_politifact.csv')
df = df.dropna(subset=['title'])
print(df.shape)
print(df['label'].value_counts())  # 1=fake, 0=real
df[['title','label']].head()


In [ ]:
# 3. Train/test split (same random_state as the local baseline, for a fair comparison)
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['title'].tolist(), df['label'].tolist(),
    test_size=0.2, random_state=42, stratify=df['label']
)
print(f"Train: {len(train_texts)} | Test: {len(test_texts)}")


In [ ]:
# 4. Tokenize with DistilBERT's tokenizer
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=64)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=64)

print("Sample tokenization:", tokenizer.decode(train_encodings['input_ids'][0]))


In [ ]:
# 5. Build PyTorch Datasets
class TitleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TitleDataset(train_encodings, train_labels)
test_dataset = TitleDataset(test_encodings, test_labels)


In [ ]:
# 6. Fine-tune DistilBERT for sequence classification
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

training_args = TrainingArguments(
    output_dir='./bert_results',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=10,
    learning_rate=2e-5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
# 7. Final evaluation + comparison to the TF-IDF baseline
results = trainer.evaluate()
print("DistilBERT results:", results)

baseline = {'accuracy': 0.877, 'f1': 0.854}  # Linear SVM baseline, from results/baseline_results.json
print(f"\nBaseline (TF-IDF + Linear SVM): Accuracy={baseline['accuracy']:.3f}, F1={baseline['f1']:.3f}")
print(f"DistilBERT:                     Accuracy={results['eval_accuracy']:.3f}, F1={results['eval_f1']:.3f}")

import json
with open('bert_results.json', 'w') as f:
    json.dump({
        'distilbert': {'accuracy': results['eval_accuracy'], 'f1': results['eval_f1']},
        'baseline_svm': baseline
    }, f, indent=2)

from google.colab import files
files.download('bert_results.json')


## Interpreting your results

- If DistilBERT beats the TF-IDF baseline: expected, since transformers
  capture context/word order that bag-of-words TF-IDF can't. Report the
  margin — on a small, short-text dataset like this (titles only, ~1000
  rows), the gap is often modest rather than dramatic.
- If DistilBERT does NOT clearly beat the baseline: also a legitimate,
  reportable result — it suggests the signal in short political headlines
  is mostly lexical (specific words/phrases), which a simpler model already
  captures well. This is worth discussing honestly rather than hidden.
- Either way, bring `bert_results.json` back to merge with your other
  results for the final write-up.
